In [1]:
import pandas as pd
import numpy as np

# Đọc dữ liệu đã chia tập, chưa tiền xử lý
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

print("\nCác cột dữ liệu:")
print(train.columns.tolist())

print("\nKiểm tra cột customer:")
print("customer" in train.columns)

Train shape: (396332, 10)
Validation shape: (98194, 10)
Test shape: (100117, 10)

Các cột dữ liệu:
['step', 'customer', 'age', 'gender', 'zipcodeOri', 'merchant', 'zipMerchant', 'category', 'amount', 'fraud']

Kiểm tra cột customer:
True


In [2]:
# Kiểm tra khoảng thời gian của từng tập
print("Train steps:", train["step"].min(), "->", train["step"].max())
print("Validation steps:", val["step"].min(), "->", val["step"].max())
print("Test steps:", test["step"].min(), "->", test["step"].max())

# Kiểm tra số lượng khách hàng
print("\nSố khách hàng:")
print("Train:", train["customer"].nunique())
print("Validation:", val["customer"].nunique())
print("Test:", test["customer"].nunique())

# Kiểm tra khách hàng xuất hiện ở nhiều tập
train_customers = set(train["customer"].unique())
val_customers = set(val["customer"].unique())
test_customers = set(test["customer"].unique())

print("\nKhách hàng xuất hiện ở cả Train và Validation:",
      len(train_customers & val_customers))

print("Khách hàng xuất hiện ở cả Train và Test:",
      len(train_customers & test_customers))

# Kiểm tra số giao dịch theo khách hàng trong Train
customer_counts = train.groupby("customer").size()

print("\nThống kê số giao dịch/khách hàng trong Train:")
print(customer_counts.describe())

Train steps: 0 -> 125
Validation steps: 126 -> 152
Test steps: 153 -> 179

Số khách hàng:
Train: 4111
Validation: 4059
Test: 4066

Khách hàng xuất hiện ở cả Train và Validation: 4058
Khách hàng xuất hiện ở cả Train và Test: 4066

Thống kê số giao dịch/khách hàng trong Train:
count    4111.000000
mean       96.407687
std        35.147443
min         2.000000
25%        83.000000
50%       114.000000
75%       120.000000
max       193.000000
dtype: float64


In [3]:
# Gộp dữ liệu theo thứ tự thời gian
all_data = pd.concat(
    [train, val, test],
    ignore_index=True
)

# Tổng hợp số giao dịch và tổng số tiền theo khách hàng ở từng step
customer_step = (
    all_data
    .groupby(["customer", "step"])
    .agg(
        tx_count=("amount", "size"),
        total_amount=("amount", "sum")
    )
    .reset_index()
    .sort_values(["customer", "step"])
)

# Tính lịch sử trước step hiện tại
customer_step["customer_tx_count"] = (
    customer_step
    .groupby("customer")["tx_count"]
    .transform(lambda x: x.cumsum().shift(1))
)

customer_step["customer_total_amount"] = (
    customer_step
    .groupby("customer")["total_amount"]
    .transform(lambda x: x.cumsum().shift(1))
)

# Tính số tiền giao dịch trung bình trong quá khứ
customer_step["customer_avg_amount"] = (
    customer_step["customer_total_amount"]
    / customer_step["customer_tx_count"]
)

# Khách hàng chưa có lịch sử: gán giá trị 0
customer_step["customer_tx_count"] = (
    customer_step["customer_tx_count"].fillna(0)
)

customer_step["customer_avg_amount"] = (
    customer_step["customer_avg_amount"].fillna(0)
)

# Gắn đặc trưng lịch sử vào từng giao dịch
history_features = customer_step[
    ["customer", "step", "customer_tx_count", "customer_avg_amount"]
]

train = train.merge(
    history_features, on=["customer", "step"], how="left"
)

val = val.merge(
    history_features, on=["customer", "step"], how="left"
)

test = test.merge(
    history_features, on=["customer", "step"], how="left"
)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

print("\nMột số dòng dữ liệu Train:")
print(train[
    ["customer", "step", "amount",
     "customer_tx_count", "customer_avg_amount"]
].head(10))

Train shape: (396332, 12)
Validation shape: (98194, 12)
Test shape: (100117, 12)

Một số dòng dữ liệu Train:
        customer  step  amount  customer_tx_count  customer_avg_amount
0  'C1093826151'     0    4.55                0.0                  0.0
1   'C352968107'     0   39.68                0.0                  0.0
2  'C2054744914'     0   26.89                0.0                  0.0
3  'C1760612790'     0   17.25                0.0                  0.0
4   'C757503768'     0   35.72                0.0                  0.0
5  'C1315400589'     0   25.81                0.0                  0.0
6   'C765155274'     0    9.10                0.0                  0.0
7   'C202531238'     0   21.17                0.0                  0.0
8   'C105845174'     0   32.40                0.0                  0.0
9    'C39858251'     0   35.40                0.0                  0.0


In [5]:
# Kiểm tra dữ liệu tại một số step sau step 0
print("Một số giao dịch tại step 1:")
print(
    train.loc[
        train["step"] == 1,
        ["customer", "step", "amount",
         "customer_tx_count", "customer_avg_amount"]
    ].head(10)
)

print("\nThống kê customer_tx_count:")
print(train["customer_tx_count"].describe())

print("\nThống kê customer_avg_amount:")
print(train["customer_avg_amount"].describe())

# Kiểm tra số giao dịch chưa có lịch sử
print(
    "\nSố giao dịch chưa có lịch sử:",
    (train["customer_tx_count"] == 0).sum()
)

# Kiểm tra giá trị thiếu
print("\nGiá trị thiếu:")
print(
    train[
        ["customer_tx_count", "customer_avg_amount"]
    ].isnull().sum()
)

Một số giao dịch tại step 1:
           customer  step  amount  customer_tx_count  customer_avg_amount
2430  'C1053477622'     1    4.65                1.0                 8.66
2431    'C77415586'     1    5.81                1.0                 6.34
2432  'C1881708420'     1   56.62                1.0                69.86
2433  'C1814870538'     1   29.32                1.0                11.13
2434  'C1155259690'     1   24.19                1.0                 3.17
2435   'C778323844'     1    6.79                1.0                23.48
2436   'C986156699'     1   74.53                1.0                20.18
2437  'C1450140987'     1  255.14                1.0               220.11
2438  'C1450140987'     1   21.93                1.0               220.11
2439  'C2055009466'     1    5.36                1.0                59.25

Thống kê customer_tx_count:
count    396332.000000
mean         54.071889
std          33.899884
min           0.000000
25%          25.000000
50%          

In [6]:
# Tham số của luật R1
MIN_HISTORY = 5
AMOUNT_MULTIPLIER = 3

def apply_rule_R1(df):
    df = df.copy()

    # Tỷ lệ số tiền hiện tại so với trung bình lịch sử
    df["amount_ratio"] = np.where(
        df["customer_avg_amount"] > 0,
        df["amount"] / df["customer_avg_amount"],
        0
    )

    # R1: Số tiền lớn hơn 3 lần trung bình lịch sử
    # và khách hàng đã có ít nhất 5 giao dịch trước đó
    df["rule_R1"] = (
        (df["customer_tx_count"] >= MIN_HISTORY) &
        (df["amount_ratio"] > AMOUNT_MULTIPLIER)
    ).astype(int)

    return df


# Áp dụng luật lên ba tập dữ liệu
train = apply_rule_R1(train)
val = apply_rule_R1(val)
test = apply_rule_R1(test)


# Kiểm tra số giao dịch bị luật R1 đánh dấu
for name, df in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    print(f"\n{name}:")
    print("Tổng giao dịch:", len(df))
    print("Giao dịch bị R1 đánh dấu:", df["rule_R1"].sum())
    print(
        "Tỷ lệ bị đánh dấu:",
        f"{df['rule_R1'].mean() * 100:.2f}%"
    )


# Xem thử các giao dịch bị R1 đánh dấu trong Train
print("\nVí dụ giao dịch bị R1 đánh dấu:")
print(
    train.loc[
        train["rule_R1"] == 1,
        ["customer", "step", "amount",
         "customer_avg_amount", "amount_ratio", "rule_R1"]
    ].head(10)
)


Train:
Tổng giao dịch: 396332
Giao dịch bị R1 đánh dấu: 10437
Tỷ lệ bị đánh dấu: 2.63%

Validation:
Tổng giao dịch: 98194
Giao dịch bị R1 đánh dấu: 2763
Tỷ lệ bị đánh dấu: 2.81%

Test:
Tổng giao dịch: 100117
Giao dịch bị R1 đánh dấu: 2834
Tỷ lệ bị đánh dấu: 2.83%

Ví dụ giao dịch bị R1 đánh dấu:
            customer  step   amount  customer_avg_amount  amount_ratio  \
6183   'C1275518867'     2  1304.93           347.210000      3.758331   
6710    'C375144053'     2  7432.48           455.522857     16.316371   
8442   'C1560676680'     3   504.97           116.643333      4.329180   
9532    'C583110837'     3  5107.70           245.458000     20.808855   
11214  'C1490048317'     4   541.86           125.121429      4.330673   
12350  'C1233802034'     5    63.69            19.874000      3.204690   
12416  'C2096265838'     5    65.36            19.488000      3.353859   
12427  'C1669649966'     5   109.50            23.942000      4.573553   
12444  'C1862169273'     5   102.88 

In [7]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Đánh giá R1 trên Validation
y_val_true = val["fraud"]
y_val_pred_R1 = val["rule_R1"]

print("===== R1 - VALIDATION =====")
print("Precision:", precision_score(y_val_true, y_val_pred_R1))
print("Recall:", recall_score(y_val_true, y_val_pred_R1))
print("F1-score:", f1_score(y_val_true, y_val_pred_R1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_true, y_val_pred_R1))

print("\nClassification Report:")
print(classification_report(y_val_true, y_val_pred_R1))

===== R1 - VALIDATION =====
Precision: 0.1885631559898661
Recall: 0.4824074074074074
F1-score: 0.2711423367161072

Confusion Matrix:
[[94872  2242]
 [  559   521]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     97114
           1       0.19      0.48      0.27      1080

    accuracy                           0.97     98194
   macro avg       0.59      0.73      0.63     98194
weighted avg       0.99      0.97      0.98     98194



In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [2, 3, 4, 5, 6]

results_R1 = []

for threshold in thresholds:
    y_pred = (
        (val["customer_tx_count"] >= MIN_HISTORY) &
        (val["amount_ratio"] > threshold)
    ).astype(int)

    results_R1.append({
        "Multiplier": threshold,
        "Precision": precision_score(val["fraud"], y_pred),
        "Recall": recall_score(val["fraud"], y_pred),
        "F1-score": f1_score(val["fraud"], y_pred),
        "Alerts": y_pred.sum()
    })

results_R1_df = pd.DataFrame(results_R1)

print(results_R1_df.to_string(index=False))

 Multiplier  Precision   Recall  F1-score  Alerts
          2   0.083922 0.572222  0.146376    7364
          3   0.188563 0.482407  0.271142    2763
          4   0.260186 0.413889  0.319514    1718
          5   0.334197 0.358333  0.345845    1158
          6   0.425121 0.325926  0.368973     828


In [9]:
# Chốt tham số R1 theo kết quả Validation
MIN_HISTORY = 5
AMOUNT_MULTIPLIER = 6

# Cập nhật lại luật R1 trên cả ba tập
train = apply_rule_R1(train)
val = apply_rule_R1(val)
test = apply_rule_R1(test)

print("Đã cập nhật R1 với hệ số:", AMOUNT_MULTIPLIER)
print("Số cảnh báo Train:", train["rule_R1"].sum())
print("Số cảnh báo Validation:", val["rule_R1"].sum())
print("Số cảnh báo Test:", test["rule_R1"].sum())

Đã cập nhật R1 với hệ số: 6
Số cảnh báo Train: 2691
Số cảnh báo Validation: 828
Số cảnh báo Test: 780


In [10]:
# Tạo bảng số giao dịch theo khách hàng và từng step
customer_step_count = customer_step[
    ["customer", "step", "tx_count"]
].copy()

# Tính số giao dịch trong 3 step trước đó
for lag in [1, 2, 3]:
    previous_step = customer_step_count[
        ["customer", "step", "tx_count"]
    ].copy()

    # Dịch step tiến lên để ghép lịch sử vào step hiện tại
    previous_step["step"] += lag
    previous_step = previous_step.rename(
        columns={"tx_count": f"tx_count_lag_{lag}"}
    )

    customer_step_count = customer_step_count.merge(
        previous_step,
        on=["customer", "step"],
        how="left"
    )

# Cộng số giao dịch ở 3 step trước
lag_columns = [
    "tx_count_lag_1",
    "tx_count_lag_2",
    "tx_count_lag_3"
]

customer_step_count["customer_tx_last_3_steps"] = (
    customer_step_count[lag_columns]
    .fillna(0)
    .sum(axis=1)
)

# Giữ lại đặc trưng R2
frequency_features = customer_step_count[
    ["customer", "step", "customer_tx_last_3_steps"]
]

# Gắn đặc trưng vào ba tập
train = train.merge(
    frequency_features,
    on=["customer", "step"],
    how="left"
)

val = val.merge(
    frequency_features,
    on=["customer", "step"],
    how="left"
)

test = test.merge(
    frequency_features,
    on=["customer", "step"],
    how="left"
)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

print("\nThống kê số giao dịch trong 3 step trước:")
print(train["customer_tx_last_3_steps"].describe())

Train shape: (396332, 15)
Validation shape: (98194, 15)
Test shape: (100117, 15)

Thống kê số giao dịch trong 3 step trước:
count    396332.000000
mean          2.738608
std           0.780770
min           0.000000
25%           3.000000
50%           3.000000
75%           3.000000
max          12.000000
Name: customer_tx_last_3_steps, dtype: float64


In [11]:
# Phân phối số giao dịch trong 3 step trước
print("Tần suất xuất hiện của từng giá trị:")
print(
    train["customer_tx_last_3_steps"]
    .value_counts()
    .sort_index()
)

# Các phân vị quan trọng
print("\nCác phân vị:")
print(
    train["customer_tx_last_3_steps"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
)

# Kiểm tra số giao dịch có tần suất từ 4 trở lên
for threshold in [4, 5, 6, 7, 8]:
    count = (train["customer_tx_last_3_steps"] >= threshold).sum()
    print(
        f"Từ {threshold} giao dịch trở lên: "
        f"{count} ({count / len(train) * 100:.2f}%)"
    )

# Xem ví dụ các giao dịch có tần suất cao
print("\nVí dụ giao dịch có tần suất từ 5 trở lên:")
print(
    train.loc[
        train["customer_tx_last_3_steps"] >= 5,
        [
            "customer", "step", "amount",
            "customer_tx_last_3_steps", "fraud"
        ]
    ].head(10)
)

Tần suất xuất hiện của từng giá trị:
customer_tx_last_3_steps
0.0      11366
1.0      12249
2.0      74895
3.0     272169
4.0      22646
5.0       2273
6.0        414
7.0        219
8.0         64
9.0         19
10.0         5
11.0         3
12.0        10
Name: count, dtype: int64

Các phân vị:
0.50    3.0
0.75    3.0
0.90    3.0
0.95    4.0
0.99    4.0
Name: customer_tx_last_3_steps, dtype: float64
Từ 4 giao dịch trở lên: 25653 (6.47%)
Từ 5 giao dịch trở lên: 3007 (0.76%)
Từ 6 giao dịch trở lên: 734 (0.19%)
Từ 7 giao dịch trở lên: 320 (0.08%)
Từ 8 giao dịch trở lên: 101 (0.03%)

Ví dụ giao dịch có tần suất từ 5 trở lên:
           customer  step   amount  customer_tx_last_3_steps  fraud
5451  'C1275518867'     2   110.90                       7.0      1
5536  'C1275518867'     2  1022.06                       7.0      1
5581  'C1685492470'     2   253.25                       5.0      1
5648  'C1275518867'     2   264.99                       7.0      1
6182  'C1275518867'     2   89

In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds_R2 = [4, 5, 6, 7, 8]

results_R2 = []

for threshold in thresholds_R2:
    y_pred = (
        val["customer_tx_last_3_steps"] >= threshold
    ).astype(int)

    results_R2.append({
        "Threshold": threshold,
        "Precision": precision_score(val["fraud"], y_pred),
        "Recall": recall_score(val["fraud"], y_pred),
        "F1-score": f1_score(val["fraud"], y_pred),
        "Alerts": y_pred.sum()
    })

results_R2_df = pd.DataFrame(results_R2)

print(results_R2_df.to_string(index=False))

 Threshold  Precision   Recall  F1-score  Alerts
         4   0.017381 0.143519  0.031006    8918
         5   0.041702 0.045370  0.043459    1175
         6   0.069869 0.014815  0.024446     229
         7   0.093023 0.007407  0.013722      86
         8   0.111111 0.000926  0.001837       9


In [14]:
# Chốt tham số R2
FREQUENCY_THRESHOLD = 5

def apply_rule_R2(df):
    df = df.copy()

    df["rule_R2"] = (
        df["customer_tx_last_3_steps"] >= FREQUENCY_THRESHOLD
    ).astype(int)

    return df


# Áp dụng R2 trên ba tập
train = apply_rule_R2(train)
val = apply_rule_R2(val)
test = apply_rule_R2(test)

print("Đã cập nhật R2 với ngưỡng:", FREQUENCY_THRESHOLD)

for name, df in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    print(f"{name}: {df['rule_R2'].sum()} cảnh báo")

Đã cập nhật R2 với ngưỡng: 5
Train: 3007 cảnh báo
Validation: 1175 cảnh báo
Test: 1378 cảnh báo


In [15]:
# Chỉ dùng giao dịch bình thường trong Train để xác định ngưỡng
normal_train_amount = train.loc[
    train["fraud"] == 0, "amount"
]

R3_AMOUNT_THRESHOLD = normal_train_amount.quantile(0.99)

print("Ngưỡng R3 (phân vị 99%):", R3_AMOUNT_THRESHOLD)

Ngưỡng R3 (phân vị 99%): 152.75270000000077


In [16]:
# Ngưỡng R3 được tính từ giao dịch bình thường trong Train
R3_AMOUNT_THRESHOLD = 152.7527

def apply_rule_R3(df):
    df = df.copy()
    df["rule_R3"] = (
        df["amount"] > R3_AMOUNT_THRESHOLD
    ).astype(int)
    return df

# Áp dụng R3 cho cả ba tập
train = apply_rule_R3(train)
val = apply_rule_R3(val)
test = apply_rule_R3(test)

print("Số cảnh báo R3:")
print("Train:", train["rule_R3"].sum())
print("Validation:", val["rule_R3"].sum())
print("Test:", test["rule_R3"].sum())

Số cảnh báo R3:
Train: 7765
Validation: 1834
Test: 1879


In [17]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_val_true = val["fraud"]
y_val_pred_R3 = val["rule_R3"]

print("R3 - Validation")
print("Precision:", precision_score(y_val_true, y_val_pred_R3))
print("Recall:", recall_score(y_val_true, y_val_pred_R3))
print("F1-score:", f1_score(y_val_true, y_val_pred_R3))
print("Confusion Matrix:")
print(confusion_matrix(y_val_true, y_val_pred_R3))

R3 - Validation
Precision: 0.4509269356597601
Recall: 0.7657407407407407
F1-score: 0.5676046671242279
Confusion Matrix:
[[96107  1007]
 [  253   827]]


In [18]:
# Tính điểm rủi ro từ ba quy tắc
for df in [train, val, test]:
    df["rule_score"] = (
        df["rule_R1"] +
        df["rule_R2"] +
        df["rule_R3"]
    )

print("Phân bố Rule Score trên Train:")
print(train["rule_score"].value_counts().sort_index())

print("\nPhân bố Rule Score trên Validation:")
print(val["rule_score"].value_counts().sort_index())

print("\nPhân bố Rule Score trên Test:")
print(test["rule_score"].value_counts().sort_index())

Phân bố Rule Score trên Train:
rule_score
0    385863
1      7509
2      2926
3        34
Name: count, dtype: int64

Phân bố Rule Score trên Validation:
rule_score
0    95227
1     2110
2      844
3       13
Name: count, dtype: int64

Phân bố Rule Score trên Test:
rule_score
0    96919
1     2368
2      821
3        9
Name: count, dtype: int64


In [19]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_val_true = val["fraud"]

results = []

for threshold in [1, 2, 3]:
    y_pred = (val["rule_score"] >= threshold).astype(int)

    results.append({
        "Rule Threshold": threshold,
        "Alerts": int(y_pred.sum()),
        "Precision": precision_score(y_val_true, y_pred),
        "Recall": recall_score(y_val_true, y_pred),
        "F1-score": f1_score(y_val_true, y_pred)
    })

rule_results = pd.DataFrame(results)

print(rule_results.round(4).to_string(index=False))

 Rule Threshold  Alerts  Precision  Recall  F1-score
              1    2967     0.2831  0.7778    0.4151
              2     857     0.4434  0.3519    0.3924
              3      13     0.6154  0.0074    0.0146


In [20]:
print("Các cột hiện tại:")
print(val.columns.tolist())

print("\nNgưỡng Rule Set:")
RULE_THRESHOLD = 1
print(RULE_THRESHOLD)

print("\nKiểm tra số lượng cảnh báo Rule Set:")
print("Train:", (train["rule_score"] >= RULE_THRESHOLD).sum())
print("Validation:", (val["rule_score"] >= RULE_THRESHOLD).sum())
print("Test:", (test["rule_score"] >= RULE_THRESHOLD).sum())

Các cột hiện tại:
['step', 'customer', 'age', 'gender', 'zipcodeOri', 'merchant', 'zipMerchant', 'category', 'amount', 'fraud', 'customer_tx_count', 'customer_avg_amount', 'amount_ratio', 'rule_R1', 'customer_tx_last_3_steps', 'rule_R2', 'rule_R3', 'rule_score']

Ngưỡng Rule Set:
1

Kiểm tra số lượng cảnh báo Rule Set:
Train: 10469
Validation: 2967
Test: 3198


In [21]:
import joblib
import pandas as pd
import numpy as np

# Nạp mô hình XGBoost đã huấn luyện
xgb_model = joblib.load("../models/xgboost.pkl")

# Nạp dữ liệu Validation đã tiền xử lý
X_val_ml = pd.read_csv("../data/processed/X_val.csv")
y_val_ml = pd.read_csv("../data/processed/y_val.csv").squeeze()

# Đọc ngưỡng XGBoost đã chọn từ Validation trước đó
with open("../models/xgboost_threshold.txt", "r") as f:
    XGB_THRESHOLD = float(f.read().strip())

# Tính xác suất gian lận
xgb_val_proba = xgb_model.predict_proba(X_val_ml)[:, 1]

print("Kích thước X_val:", X_val_ml.shape)
print("Số xác suất dự đoán:", len(xgb_val_proba))
print("Ngưỡng XGBoost:", XGB_THRESHOLD)
print("Ngưỡng Rule Set:", RULE_THRESHOLD)

Kích thước X_val: (98194, 79)
Số xác suất dự đoán: 98194
Ngưỡng XGBoost: 0.97
Ngưỡng Rule Set: 1


In [22]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Nhãn thực tế
y_true = y_val_ml.astype(int).values

# Dự đoán XGBoost
xgb_pred = (xgb_val_proba >= XGB_THRESHOLD).astype(int)

# Dự đoán Rule Set
rule_pred = (val["rule_score"].values >= RULE_THRESHOLD).astype(int)

# Phương án 1: XGBoost độc lập
hybrid_or_pred = np.logical_or(
    xgb_pred, rule_pred
).astype(int)

# Phương án 2: XGBoost AND Rule Set
hybrid_and_pred = np.logical_and(
    xgb_pred, rule_pred
).astype(int)

# Hàm đánh giá
def evaluate_prediction(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Alerts": int(y_pred.sum()),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred)
    }

results = []

results.append(
    evaluate_prediction(y_true, xgb_pred, "XGBoost")
)

results.append(
    evaluate_prediction(y_true, hybrid_or_pred, "Hybrid OR")
)

results.append(
    evaluate_prediction(y_true, hybrid_and_pred, "Hybrid AND")
)

hybrid_results = pd.DataFrame(results)

print(hybrid_results.round(4).to_string(index=False))

     Model  Alerts  Precision  Recall  F1-score
   XGBoost     986     0.8834  0.8065    0.8432
 Hybrid OR    3087     0.2967  0.8481    0.4396
Hybrid AND     866     0.9180  0.7361    0.8171


In [23]:
print("XGBoost:")
print(confusion_matrix(y_true, xgb_pred))

print("\nHybrid OR:")
print(confusion_matrix(y_true, hybrid_or_pred))

print("\nHybrid AND:")
print(confusion_matrix(y_true, hybrid_and_pred))

XGBoost:
[[96999   115]
 [  209   871]]

Hybrid OR:
[[94943  2171]
 [  164   916]]

Hybrid AND:
[[97043    71]
 [  285   795]]


In [24]:
# Chuẩn hóa điểm Rule Set về khoảng 0 đến 1
rule_score_normalized = val["rule_score"].values / 3

# Các trọng số thử nghiệm
alphas = [0.1, 0.2, 0.3, 0.4, 0.5]

hybrid_score_results = []

for alpha in alphas:

    hybrid_score = (
        (1 - alpha) * xgb_val_proba
        + alpha * rule_score_normalized
    )

    # Tạm sử dụng ngưỡng XGBoost đã chọn
    hybrid_pred = (
        hybrid_score >= XGB_THRESHOLD
    ).astype(int)

    hybrid_score_results.append({
        "Alpha": alpha,
        "Alerts": int(hybrid_pred.sum()),
        "Precision": precision_score(y_true, hybrid_pred),
        "Recall": recall_score(y_true, hybrid_pred),
        "F1-score": f1_score(y_true, hybrid_pred)
    })

hybrid_score_df = pd.DataFrame(hybrid_score_results)

print(hybrid_score_df.round(4).to_string(index=False))

 Alpha  Alerts  Precision  Recall  F1-score
   0.1       8     1.0000  0.0074    0.0147
   0.2       8     1.0000  0.0074    0.0147
   0.3       8     1.0000  0.0074    0.0147
   0.4       8     1.0000  0.0074    0.0147
   0.5       9     0.8889  0.0074    0.0147


In [25]:
alphas = [0.1, 0.2, 0.3, 0.4, 0.5]
thresholds = np.arange(0.05, 1.00, 0.01)

hybrid_tuning_results = []

for alpha in alphas:

    hybrid_score = (
        (1 - alpha) * xgb_val_proba
        + alpha * rule_score_normalized
    )

    best_f1 = -1
    best_result = None

    for threshold in thresholds:

        hybrid_pred = (
            hybrid_score >= threshold
        ).astype(int)

        precision = precision_score(
            y_true, hybrid_pred, zero_division=0
        )

        recall = recall_score(
            y_true, hybrid_pred, zero_division=0
        )

        f1 = f1_score(
            y_true, hybrid_pred, zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1

            best_result = {
                "Alpha": alpha,
                "Threshold": threshold,
                "Alerts": int(hybrid_pred.sum()),
                "Precision": precision,
                "Recall": recall,
                "F1-score": f1
            }

    hybrid_tuning_results.append(best_result)

hybrid_tuning_df = pd.DataFrame(
    hybrid_tuning_results
)

print(
    hybrid_tuning_df.round(4).to_string(index=False)
)

 Alpha  Threshold  Alerts  Precision  Recall  F1-score
   0.1       0.88    1004     0.8725  0.8111    0.8407
   0.2       0.78    1047     0.8500  0.8241    0.8369
   0.3       0.68    1082     0.8364  0.8380    0.8372
   0.4       0.58    1109     0.8206  0.8426    0.8314
   0.5       0.49    1083     0.8199  0.8222    0.8211


In [27]:
# Cấu hình Hybrid được lựa chọn từ kết quả Validation
BEST_ALPHA = 0.1
BEST_HYBRID_THRESHOLD = 0.88

print("Best Alpha:", BEST_ALPHA)
print("Best Hybrid Threshold:", BEST_HYBRID_THRESHOLD)

Best Alpha: 0.1
Best Hybrid Threshold: 0.88


In [29]:
# Nạp dữ liệu Test
X_test_ml = pd.read_csv("../data/processed/X_test.csv")
y_test_ml = pd.read_csv("../data/processed/y_test.csv").squeeze()

# Dự đoán xác suất bằng XGBoost
xgb_test_proba = xgb_model.predict_proba(X_test_ml)[:, 1]

# Chuẩn hóa điểm Rule Set
rule_score_test_normalized = test["rule_score"].values / 3

# Tính điểm Hybrid
hybrid_test_score = (
    (1 - BEST_ALPHA) * xgb_test_proba
    + BEST_ALPHA * rule_score_test_normalized
)

# Dự đoán XGBoost
xgb_test_pred = (
    xgb_test_proba >= XGB_THRESHOLD
).astype(int)

# Dự đoán Hybrid
hybrid_test_pred = (
    hybrid_test_score >= BEST_HYBRID_THRESHOLD
).astype(int)

print("X_test:", X_test_ml.shape)
print("Số dự đoán XGBoost:", len(xgb_test_pred))
print("Số dự đoán Hybrid:", len(hybrid_test_pred))

X_test: (100117, 79)
Số dự đoán XGBoost: 100117
Số dự đoán Hybrid: 100117


In [30]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Nhãn thực tế
y_test_true = y_test_ml.astype(int).values

# Đánh giá XGBoost
models_test = {
    "XGBoost": {
        "pred": xgb_test_pred,
        "score": xgb_test_proba
    },
    "Hybrid": {
        "pred": hybrid_test_pred,
        "score": hybrid_test_score
    }
}

final_results = []

for model_name, data in models_test.items():

    y_pred = data["pred"]
    y_score = data["score"]

    final_results.append({
        "Model": model_name,
        "Precision": precision_score(y_test_true, y_pred),
        "Recall": recall_score(y_test_true, y_pred),
        "F1-score": f1_score(y_test_true, y_pred),
        "PR-AUC": average_precision_score(
            y_test_true, y_score
        ),
        "Alerts": int(y_pred.sum())
    })

final_results_df = pd.DataFrame(final_results)

print("FINAL TEST RESULTS")
print(final_results_df.round(4).to_string(index=False))

FINAL TEST RESULTS
  Model  Precision  Recall  F1-score  PR-AUC  Alerts
XGBoost     0.8932  0.7824    0.8342  0.9213     946
 Hybrid     0.8805  0.7981    0.8373  0.9053     979


In [31]:
print("Confusion Matrix - XGBoost:")
print(confusion_matrix(y_test_true, xgb_test_pred))

print("\nConfusion Matrix - Hybrid:")
print(confusion_matrix(y_test_true, hybrid_test_pred))

Confusion Matrix - XGBoost:
[[98936   101]
 [  235   845]]

Confusion Matrix - Hybrid:
[[98920   117]
 [  218   862]]
